In [ ]:
'''
This code processes housing data and predicts renewal 
probabilities.
'''

# Imports
import json
import sqlite3
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import matplotlib.pyplot as plt

In [21]:
# Functions to be used to process data

''' 
Takes a dataframe and outputs a column where the dates are turned 
into a numerical representation where years are integers and days 
are decimals past that integer. For missing date entries, NaN is 
entered. NaN entried have to be dealt with later.

args:
    df (pd): Dataframe
    col (str): key to date column to turn into numerics
returns:
    dates_numeric (pd): Dataframe column of numerics
'''
def dates_to_numerics(df, col):
    dates_numeric = pd.to_datetime(df[col], errors = 'coerce')
    dates_numeric = dates_numeric.dt.year + (dates_numeric.dt.dayofyear - 1) / 365.25 + dates_numeric.dt.hour / 8760 + dates_numeric.dt.minute / 525600
    #dates_numeric = dates_numeric.fillna(dates_numeric.median())
    return dates_numeric

''' 
Takes a dataframe and outputs a one hot encoding of a desired
column. Returns the whole dataframe with new one hot encoded
columns as well as names of one hot columns.

args:
    df (pd): Dataframe
    col (str): key to date column to turn into numerics
returns:
    df (pd): Full dataframe with one hot columns
    encoded_cols ([str]): names of one-hot encoded columns
'''
def one_hot_embedding(df, col):
    encoder = OneHotEncoder(drop = 'first', sparse_output = False)
    encoded_data = encoder.fit_transform(df[[col]])
    encoded_cols = encoder.get_feature_names_out([col])
    encoded_df = pd.DataFrame(encoded_data, columns = encoded_cols, index = df.index)

    return pd.concat([df.drop(col, axis = 1), encoded_df], axis = 1), encoded_cols

''' 
Identifies outliers using the Interquartile Range (IQR) method
and replaces outliers using winsorization.

args:
    series (pd): Dataframe column
    thr (float): defines the IQR bound, 1.5 is standard
returns:
    (array) : Column with outliers replaced
'''
def winsorization_iqr(series, thr = 1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = 1.5 * IQR + Q3
    lower_bound = Q1 - 1.5 * IQR
    return np.clip(series, lower_bound, upper_bound) 

''' 
Create a dataframe from db and json file. Does some initial
processing of the json file as follows:
    - Counts the number of messages each user sends.
These features are appended onto the residents information.
args:
    db_path (str): path to db file
    json_path (str): path to json file
returns:
    residents (df)
    maintenance (df)
    offers (df)
'''
def read_data(db_path, json_path, is_train = True):
    # Reads either test or training data
    train = "_train" if is_train else "_test"

    # Parse the .db file
    conn = sqlite3.connect(db_path)
    residents = pd.read_sql_query(f"SELECT * FROM residents{train}", conn)
    maintenance = pd.read_sql_query(f"SELECT * FROM maintenance_history{train}", conn)
    offers = pd.read_sql_query(f"SELECT * FROM renewal_offers{train}", conn)

    # Parse the json file
    with open(json_path, 'r') as f:
        chats = json.load(f)

    chat_features = []
    for res in chats:
        res_id = int(res['resident_id'])
        # Count the number of messages the user sends
        user_msgs = [[m['message'] for m in res['conversation'] if m['role'] == 'user']]
        msg_count = len(user_msgs)
        chat_features.append({'resident_id' : res_id, 'msg_count' : msg_count})
    chat_df = pd.DataFrame(chat_features)

    # Merge chat data with residents data
    residents = residents.merge(chat_df, on = 'resident_id', how = 'left')
    residents['msg_count'] = residents['msg_count'].fillna(0)

    return residents, maintenance, offers, chats

In [105]:
# Read in the data

residents_train, maintenance_train, offers_train, chats_train = read_data(
    "research-practical-main/data/resident_database.db", 
    "research-practical-main/data/conversations_train.json", True)

residents_test, maintenance_test, offers_test, chats_test = read_data(
    "research-practical-main/data/resident_database.db", 
    "research-practical-main/data/conversations_test.json", False)

In [109]:
# Preprocess the Data

''' 
Takes housing data in terms of dataframes and preprocesses them
according to the following specifications.

args:
    residents (df): Residents data
    maintenance (df): Maintenance data
    offers (df): Renewal Offer data
returns:
    final_data (pd): Processed data removing data that won't be
                     used.
'''

def preprocess_data(residents, maintenance, offers, is_train = True):
    ### Pet Details
    # First character of 'pet_details' is the number of pets. Throw away pet information 
    # and keep number of pets. Replace None entries with 0.
    residents['pet_details'] = residents['pet_details'].str.strip().str[0]
    residents['pet_details'] = pd.to_numeric(residents['pet_details'])
    residents['pet_details'] = residents['pet_details'].fillna(0)

    ### Referrals
    # Use one hot embedding for 'referrals'
    residents, referral_onehot = one_hot_embedding(residents, 'referral')

    ### ADA Support
    # Turn into binary deciding if there is ADA support
    residents['ADA_support'] = residents['ADA_support']
    residents['ADA_support'] = residents['ADA_support'].notnull().astype(int)

    ### Dates
    # Turn all date columns into numerics where integers are years and decimals are portions of years
    # Start by defining the columns to be numericalized
    dates_res = ['lease_start_date', 'lease_end_date']
    dates_maint = ['date_submitted', 'target_completion_date', 'date_completed']
    dates_offers = ['date_sent']
    # Add numerical date columns
    for col in dates_res:
        residents[col + "_num"] = dates_to_numerics(residents, col)
    for col in dates_maint:
        maintenance[col + "_num"] = dates_to_numerics(maintenance, col)
    for col in dates_offers:
        offers[col + "_num"] = dates_to_numerics(offers, col)

    ### Target Completion Date and Completion Date
    # Compute the time it takes to complete a request and the difference in expected completion time
    # Make a new column for whether job is completed based on 'status'
    maintenance['completion_time'] = maintenance['target_completion_date_num'] - maintenance['date_submitted_num']
    maintenance['completion_diff'] = maintenance['target_completion_date_num'] - maintenance['date_completed_num']
    maintenance['is_completed'] = (maintenance['status'] == 'completed').astype(int)

    # Combine completion times and differences based on resident. Total is used because this will be averaged over 'unit_number' later.
    maintenance_agg = maintenance.groupby(['resident_id']).agg(
    tot_completion_time = ('completion_time', 'sum'),
    tot_completion_diff = ('completion_diff', 'sum'),
    number_of_requests = ('resident_id', 'size'),
    number_of_completed_requests = ('is_completed', 'sum')).reset_index()

    # NaNs for completion times which are not inputted. There is a lot of missing completion times.
    # NaNs will be a problem later when calculating averages. For now, fill them in with the median.
    maintenance_agg['tot_completion_diff'] = maintenance_agg['tot_completion_diff'].fillna(maintenance_agg['tot_completion_diff'].median())
    maintenance_agg['tot_completion_time'] = maintenance_agg['tot_completion_time'].fillna(maintenance_agg['tot_completion_time'].median())

    ### Combine all data sets
    # Combining residents and maintenance data after aggregating all the 'resident_id' data
    aggregated_data = residents.merge(maintenance_agg, on = 'resident_id', how = 'left')
    # Combine offers data based on 'unit_number'. This leaves a lot of cloned data for multiple people households.    
    aggregated_data = aggregated_data.merge(offers, left_on = 'unit_number', right_on = 'unit')
    # Clean up the missing entries in the maintenance data
    aggregated_data['number_of_requests'] = aggregated_data['number_of_requests'].fillna(0)
    aggregated_data['number_of_completed_requests'] = aggregated_data['number_of_completed_requests'].fillna(0)

    ### Occupancy Data
    # Add a column for number of occupants by adding in repeat 'unit_number' and 'lease_end_date' 
    aggregated_data['occupancy_count'] = aggregated_data.groupby(['unit_number', 'lease_end_date'])['unit_number'].transform('size')

    ### Aggregate into one data set by 'unit_number' and 'lease_end_date'
    # Define the aggregation rules keeping the features we deem important
    agg_rules = {
        'age' : 'mean',
        'lease_start_date_num' : 'first',
        'lease_end_date_num' : 'first',
        'pet_details' : 'max',
        'ADA_support' : 'max',
        'newsletter_opt_in' : 'sum', # sum incase it matters that many household members are opted in
        'monthly_rent' : 'first',
        'referral_craigslist' : 'sum',
        'referral_ex-resident' : 'sum',
        'referral_facebook' : 'sum',
        'referral_resident' : 'sum',
        'referral_walk-in' : 'sum',
        'referral_zillow' : 'sum',
        'referral_None' : 'sum',
        'msg_count' : 'sum',
        'tot_completion_time' : 'sum',
        'tot_completion_diff' : 'sum',
        'number_of_requests' : 'sum',
        'number_of_completed_requests' : 'sum',
        'occupancy_count' : 'first',
        'rent' : 'first',
        'date_sent_num' : 'first',
    }
    if is_train:
        agg_rules['renewal_decision'] = 'first'
    final_data = aggregated_data.groupby(['unit_number', 'lease_end_date'], as_index = False).agg(agg_rules)

    ### Define New Features for Final Dataset
    final_data['avg_completion_time'] = final_data['tot_completion_time'] / final_data['number_of_completed_requests']
    final_data['avg_completion_diff'] = final_data['tot_completion_diff'] / final_data['number_of_completed_requests']
    # For no completed requests, replace by the median
    final_data['avg_completion_time'].replace([np.inf, -np.inf], np.nan, inplace=True)
    final_data['avg_completion_diff'].replace([np.inf, -np.inf], np.nan, inplace=True)
    final_data['avg_completion_time'] = final_data['avg_completion_time'].fillna(final_data['avg_completion_time'].median())
    final_data['avg_completion_diff'] = final_data['avg_completion_time'].fillna(final_data['avg_completion_diff'].median())
    final_data['percent_completion'] = final_data['number_of_completed_requests'] / final_data['number_of_requests']
    final_data['rent_diff'] = final_data['rent'] - final_data['monthly_rent']
    final_data['percent_rent_diff'] = final_data['rent_diff'] / final_data['monthly_rent']
    final_data['lease_length'] = final_data['lease_end_date_num'] - final_data['lease_start_date_num']
    final_data['lease_renew_window'] = final_data['lease_end_date_num'] - final_data['date_sent_num']

    ### Handle Outliers
    final_data['percent_rent_diff'] = winsorization_iqr(final_data['percent_rent_diff'])

    return final_data

In [110]:
final_data_train = preprocess_data(residents_train.copy(), maintenance_train.copy(), offers_train.copy())
final_data_test = preprocess_data(residents_test.copy(), maintenance_test.copy(), offers_test.copy(), is_train = False)

/var/folders/q4/jd5whcss467g513yxycgxn8h0000gn/T/ipykernel_25050/965859984.py:113: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_data['avg_completion_time'].replace([np.inf, -np.inf], np.nan, inplace=True)
/var/folders/q4/jd5whcss467g513yxycgxn8h0000gn/T/ipykernel_25050/965859984.py:114: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object

In [ ]:
### Visualize the Data
(final_data_train['percent_rent_diff'] * 100).hist(bins=20, edgecolor='black')
plt.title('Histogram of percentage rent increase')
plt.xlabel('Percentage Rent Increase')
plt.ylabel('Frequency')
plt.savefig("figures/hist_perc_rent_incr.png")
plt.clf()

(final_data_train['avg_completion_time'] * 365.25).hist(bins=20, edgecolor='black')
plt.title('Histogram of Maintenance Completion Time')
plt.xlabel('Completion Time (Days)')
plt.ylabel('Frequency')
plt.savefig("figures/hist_avg_comp_time.png")
plt.clf()

(final_data_train['avg_completion_diff'] * 365.25).hist(bins=20, edgecolor='black')
plt.title('Histogram of Maintenance Completion Difference')
plt.xlabel('Completion Difference (Days)')
plt.ylabel('Frequency')
plt.savefig("figures/hist_avg_comp_diff.png")
plt.clf()

(final_data_train['number_of_requests']).hist(bins=10, edgecolor='black')
plt.title('Histogram of Number of Maintenance Requests')
plt.xlabel('Number of Requests')
plt.ylabel('Frequency')
plt.savefig("figures/hist_number_of_requests.png")
plt.clf()

(final_data_train['percent_completion'] * 100).hist(bins=10, edgecolor='black')
plt.title('Histogram of Percent Completed Maintenance')
plt.xlabel('Completion Percent')
plt.ylabel('Frequency')
plt.savefig("figures/hist_perc_completed_requests.png")
plt.clf()

(final_data_train['lease_length']).hist(bins=10, edgecolor='black')
plt.title('Histogram of Lease Length')
plt.xlabel('Lease Length (years)')
plt.ylabel('Frequency')
plt.savefig("figures/hist_lease_length.png")
plt.clf()

(final_data_train['lease_renew_window']).hist(bins=10, edgecolor='black')
plt.title('Histogram of Lease Renewal Window')
plt.xlabel('Lease Renewal Window (years)')
plt.ylabel('Frequency')
plt.savefig("figures/hist_lease_renew_window.png")
plt.clf()

print(final_data_train['occupancy_count'].value_counts())
print(final_data_train['pet_details'].value_counts())
print(final_data_train['msg_count'].value_counts())

occupancy_count
1    220
2    164
4     45
3     25
Name: count, dtype: int64
pet_details
1.0    213
0.0    154
2.0     87
Name: count, dtype: int64
msg_count
1    220
2    164
4     45
3     25
Name: count, dtype: int64


<Figure size 640x480 with 0 Axes>

In [111]:
print(final_data_train.keys())

Index(['unit_number', 'lease_end_date', 'age', 'lease_start_date_num',
       'lease_end_date_num', 'pet_details', 'ADA_support', 'newsletter_opt_in',
       'monthly_rent', 'referral_craigslist', 'referral_ex-resident',
       'referral_facebook', 'referral_resident', 'referral_walk-in',
       'referral_zillow', 'referral_None', 'msg_count', 'tot_completion_time',
       'tot_completion_diff', 'number_of_requests',
       'number_of_completed_requests', 'occupancy_count', 'rent',
       'date_sent_num', 'renewal_decision', 'avg_completion_time',
       'avg_completion_diff', 'percent_completion', 'rent_diff',
       'percent_rent_diff', 'lease_length', 'lease_renew_window'],
      dtype='object')


In [122]:
### Setting up the Model ###

# Functions to Support Model Set Up
''' 
Calculates the Variance Inflation Factor for a variety of 
features in order to identify colinearity in the feature set.

args:
    df (pd): Input features
    features ([str]): names of features
returns:
    vif ([float]) : VIF values for each feature
'''
def calc_vif(df, features):
    vif = []
    for feat in features:
        X_test = df[features].drop(columns = [feat])
        Y_test = df[feat]
        vif_model = LinearRegression()
        vif_model.fit(X_test, Y_test)
        r2 = vif_model.score(X_test, Y_test)
        if r2 > 0.999:
            vif.append(10000)
        else:
            vif.append(1/(1-r2))
    return vif

''' 
Some tests in order to prune the feature space to find the
most important features
args:
    init_features ([str]): Input features
    X (df) : Input features
    Y (df) : Target features
returns:
    features ([str]) : Most important features to keep
'''
def prune_features(init_features, model, X_train, Y_train, X_val, Y_val, imp_thr = 1e-3):
    features = []

    # Do a VIF analysis first and keep the ones with low VIF
    vif = calc_vif(X_train[init_features], init_features)
    print("--- Features with High Colinearity ---")
    for iv, v in enumerate(vif):
        if v > 5:
            print(init_features[iv])
    # features = [feat for feat, value in zip(init_features, vif) if value < 5]

    # Permutation importance 
    perm_importance = permutation_importance(model, X_val, Y_val, n_repeats=10, random_state=42)
    importance_scores = perm_importance.importances_mean
    print("--- Importance Scores ---")
    for i in range(len(importance_scores)):
        if abs(importance_scores[i]) > imp_thr:
            features.append(X_train.columns[i])
            print(f"{X_train.columns[i]}: {importance_scores[i]:.4f}")
        else:
            print(f"{X_train.columns[i]}: {importance_scores[i]:.4f} - DISCARDED")


            
    return features


In [116]:
# First we define the features to be used in the model among
# all the features in the dataset.

init_features = ['age', 'pet_details', 'ADA_support', 'newsletter_opt_in',
            'monthly_rent', 'percent_rent_diff', 'msg_count', 
            'number_of_requests', 'percent_completion', 
            'occupancy_count', 'avg_completion_time', 
            'avg_completion_diff', 'lease_length', 
            'lease_renew_window',
            'referral_craigslist', 'referral_ex-resident',
            'referral_facebook', 'referral_resident', 'referral_walk-in',
            'referral_zillow', 'referral_None']


X_train = final_data_train[init_features]
Y_train = final_data_train['renewal_decision']
X_test = final_data_test[init_features]


In [ ]:
n_train = int(X_train.shape[0] * 0.9)
X_ptrain = X_train[:n_train]
Y_ptrain = Y_train[:n_train]
X_val = X_train[n_train:]
Y_val = Y_train[n_train:]

# Test a Linear Regression Model
model_lr = LinearRegression().fit(X_ptrain, Y_ptrain)
print(final_data_test.shape[0])
features_linear_regression = prune_features(init_features, model_lr, X_ptrain, Y_ptrain, X_val, Y_val)

114
--- Features with High Colinearity ---
msg_count
occupancy_count
avg_completion_time
avg_completion_diff
--- Importance Scores ---
age: 0.2408
pet_details: -0.0113
ADA_support: -0.0017
newsletter_opt_in: -0.0010 - DISCARDED
monthly_rent: -0.0073
percent_rent_diff: 0.0044
msg_count: 0.0147
number_of_requests: 0.0838
percent_completion: 0.0025
occupancy_count: 0.0147
avg_completion_time: 0.0013
avg_completion_diff: 0.0013
lease_length: 0.0491
lease_renew_window: -0.0017
referral_craigslist: 0.0022
referral_ex-resident: 0.0022
referral_facebook: 0.0083
referral_resident: 0.0027
referral_walk-in: -0.0180
referral_zillow: -0.0060
referral_None: 0.0041


In [ ]:
# Test for 